In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False, json.loads(sys.argv[1])['city'],  json.loads(sys.argv[1])['scenario'])

# START

In [2]:
import sys
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import stepmodel, connectionscanmodel
from quetzal.io import excel
import os
import numpy as np
import pandas as pd
if MANUAL:
    %matplotlib inline

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'models/{}/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

In [5]:
if var['run_mode']['mode'] != 'timetables':
    sm = stepmodel.read_zippedpickles(mpath + r'generation_distribution')
else:
    sm = connectionscanmodel.read_zippedpickles(mpath + r'generation_distribution')

zone_to_road: 100%|██████████| 21/21 [00:02<00:00,  9.66it/s]


## MaP

In [6]:
sm.los = sm.los[sm.los["route_type"]!="walk"]
sm.segments = ["volume"]
sm.volumes
# TODO Louison: enlever une part fixe de marche à pied en fonction de la distance entre zones

,origin,destination,volume
0,zone_0,zone_0,0.127052
1,zone_0,zone_1,0.165520
2,zone_0,zone_2,0.013332
3,zone_0,zone_3,0.011725
4,zone_0,zone_4,0.006879
...,...,...,...
4619,zone_67,zone_63,0.008827
4620,zone_67,zone_64,0.007701
4621,zone_67,zone_65,0.090092
4622,zone_67,zone_66,0.026618


## Configure Logit scales

In [7]:
var['preparation_logit'].to_dict()

{'time': -0.00033860092781289584,
 'price': -0.000334658514016994,
 'transfers': -0.5,
 'mode': 1,
 'pt_mode': 0.8,
 'pt_path': 0.5}

In [8]:
kwargs = var['preparation_logit'].to_dict()

sm.preparation_logit(
    **kwargs, segments=['volume'],
    route_types=['root', 'car', 'bike', 'pt'] + list(sm.links["route_type"].unique())
)

In [9]:
sm.utility_values.T

value,time,price,ntransfers,mode_utility
segment,,,,
root,-0.000339,-0.000335,-0.5,1.0
volume,-0.000339,-0.000335,-0.5,1.0


In [10]:
sm.logit_scales.T

route_type,root,car,bike,pt,bus,rail,tramway
segment,,,,,,,
root,1.0,0.0,0.5,0.8,0.5,0.5,0.5
volume,1.0,0.0,0.5,0.8,0.5,0.5,0.5


In [11]:
sm.mode_nests.loc['bike'] = 'root'
sm.mode_nests

segment,root,volume
route_type,,
root,NaN,NaN
car,root,root
bike,root,root
pt,root,root
bus,pt,pt
rail,pt,pt
tramway,pt,pt


In [12]:
# TODO: ajouter une valeur par défaut pour les paramètres de mode TC manquants ?
for mode in set(sm.mode_utility.index) - {"root", "pt"}:
    sm.mode_utility.loc[mode] = var['mode_utility'][mode]

sm.mode_utility

segment,root,volume
route_type,,
root,0.0,0.0
car,1.0,1.0
bike,0.5,0.5
pt,0.0,0.0
bus,0.0,0.0
rail,0.0,0.0
tramway,0.0,0.0


## Utility

In [13]:
sm.los['price'] = sm.los['fare']
# TODO Louison: update time PT avec la pondération des temps de marche, attente et transfer (cf note technique)
# sm.los['time'] = sm.los['gtime']

# TODO : BOUCLE à crée avec les cellules dessous, à mettre au propre dans des fonctions avant

Principe: affectation par tranche et mise à jour des LOS
on découpe la demande en 10 (paramétrable) tranches et on fait 10 itérations:

- a. Logit
- b. on fait l’affectation TC et on met à jour les LOS TC
- c. on fait l’affectation CAR (AON) et on met à jour les LOS CAR
- d. on fusionne tous les LOS mis à jour et on passe à la tranche suivante

Les tranches sont de taille différente (plus grosses au début)

## A. RUN LOGIT

In [14]:
# Calculate the utilities and the probabilities
sm.analysis_mode_utility(how='mean')
sm.initialize_logit()
sm.step_logit()
sm.compute_los_volume()

100%|██████████| 1/1 [00:00<00:00, 104.97it/s]


100%|██████████| 10/10 [00:00<00:00, 26.99it/s]


# B. PT ASSIGNMENT

In [15]:
sm.pt_los = sm.los[~(sm.los['route_type'].isin(['walk', 'car', 'bike']))]

# PT ASSIGNMENT
sm.step_pt_assignment(
    on_road_links=False, boardings=True, alightings=True, transfers=True
)

# compute TC congested gtime
alpha = 0.15
beta = 4
sm.links["capacity"] = 0.1 # TODO Louison: mettre la capacité des liens TC
sm.links["gtime"] = sm.links["time"] * (1 + alpha * sm.links["load"] / sm.links["capacity"]) ** beta

# update pt_los with congested time
d = sm.links["gtime"].to_dict()
sm.pt_los['in_vehicle_time'] = sm.pt_los['link_path'].apply(lambda ls: sum([d[t] for t in ls]))
cols = [
    'access_time',
    'footpath_time',
    'waiting_time',
    'boarding_time',
    'in_vehicle_time',
]
sm.pt_los['time'] = sm.pt_los[cols].T.sum()

volume: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]


# C. CAR ASSIGNMENT

In [16]:
sm.road_links["capacity"] = 1

In [17]:
sm.zone_to_road["length"] = sm.zone_to_road.geometry.apply(lambda g: g.length)

In [ ]:
car = sm.copy()
CAR_OCCUPANCY_RATE = 1.5  # TODO Louison: valeur de l'occupation moyenne des voitures 
# CARS
car_los = sm.los.loc[sm.los['pathfinder_session']=='car_aon']
car_los['volume'] *= 1 / CAR_OCCUPANCY_RATE
car.volumes = car_los[["origin", "destination", "volume"]]

# car.step_road_pathfinder(method='aon', segments=["volume"])
car.step_road_pathfinder(method='msa', segments=["volume"], tolerance=10) # pour les itérations intermédiaires, pas besoin d’un relgap à 0.01%

# TODO Louison: remplacer roadpathfinder MSA par un simple AON + mise à jour des temps sur les road_links à partir des courbes BPR T0*(1+\a * (T/C)**\b)

# D. MERGE

In [ ]:
#TODO Louison: merge update car los, pt-los, with the rest (bike)
#TODO Louison: back to logit (boucle)

# TOZIPPEDPICKLES

In [352]:
# TODO: lighten los

In [353]:
sm.to_zippedpickles(
    mpath + '{scen}/assignment'.format(scen=SCENARIO), 
)

loaded_nodes: 100%|██████████| 42/42 [00:05<00:00,  7.44it/s]   


In [354]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

# CONTROL

In [355]:
temp = sm.los[sm.los['origin']==sm.los['destination']]
round(
    temp.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() /\
    temp[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() * 100
)

route_type
car            7.0
motorcycle     5.0
walk          88.0
dtype: float64

In [356]:
round(
    sm.los.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() /\
    sm.los[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() * 100
)

route_type
car                  6.0
motorcycle           5.0
paratransit_pt      26.0
paratransit_taxi     1.0
walk                62.0
dtype: float64

In [357]:
round(
    sm.los.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum() /\
    sm.los[['pt_captive', 'car_owners', 'moto_owners']].sum() * 100
)

,pt_captive,car_owners,moto_owners
route_type,,,
car,0.0,76.0,0.0
motorcycle,0.0,0.0,86.0
paratransit_pt,30.0,4.0,0.0
paratransit_taxi,1.0,0.0,0.0
walk,69.0,20.0,14.0
